# Final Project — Experimentation, A/B Testing & Causal Inference

## Improving an Online Learning Website

**Scenario:** An online learning platform wants to know whether a simplified landing page with a clearer registration call-to-action increases user registrations.

> **Data note:** This project uses a **simulated A/B-test dataset** created for educational purposes. The dataset is included in this repository as `ab_test_data.csv`.


## 1. Problem Definition

### Business problem
The platform receives many landing-page visitors, but only a limited share register for an account. The team wants to test whether simplifying the landing page and making the registration button more prominent improves registration conversion.

### Business question
Does the redesigned landing page increase the percentage of visitors who register?

### Hypothesis
- **H0 (null):** The Treatment conversion rate is equal to the Control conversion rate.
- **H1 (alternative):** The Treatment conversion rate is different from the Control conversion rate.

### Target users
New visitors who reach the landing page and are eligible to register.

### Control
The current landing page.

### Treatment
A simplified landing page with:
- a clearer value proposition,
- a more prominent registration CTA,
- fewer distractions,
- a shorter registration path.


## 2. Experiment Design

### Primary metric (OEC)
**Registration conversion rate**

\[
Conversion\ Rate = \frac{Registered\ Users}{Total\ Users}
\]

### Guardrail metric
**Page error rate.** The redesigned experience should not materially increase technical errors.

Guardrail threshold: Treatment page-error rate should not increase by more than **0.5 percentage points** versus Control.

### Random assignment
Eligible users are randomly assigned **50/50 at the user level** to Control or Treatment. Assignment should remain persistent for the user during the experiment.

### Practical threshold / MDE
The Minimum Detectable / practically meaningful Effect is defined as:

**+1.5 percentage points absolute increase in registration conversion.**

With a 12% baseline, this corresponds to approximately a 12.5% relative increase.


## 3. A/B Test Analysis

In [1]:
import pandas as pd
import numpy as np
import math
from scipy.stats import norm

df = pd.read_csv("ab_test_data.csv")
df.head()


,user_id,group,registered,page_error
0,1,Control,0,0
1,2,Control,0,0
2,3,Control,1,0
3,4,Control,0,0
4,5,Control,0,0


In [2]:
summary = df.groupby("group").agg(
    users=("user_id", "count"),
    registrations=("registered", "sum"),
    conversion_rate=("registered", "mean"),
    page_errors=("page_error", "sum"),
    page_error_rate=("page_error", "mean")
).reset_index()

summary


,group,users,registrations,conversion_rate,page_errors,page_error_rate
0,Control,10000,1200,0.120,100,0.010
1,Treatment,10000,1450,0.145,110,0.011


In [3]:
control = summary.loc[summary["group"] == "Control"].iloc[0]
treatment = summary.loc[summary["group"] == "Treatment"].iloc[0]

p_control = control["conversion_rate"]
p_treatment = treatment["conversion_rate"]

absolute_lift = p_treatment - p_control
relative_lift = absolute_lift / p_control

print(f"Control conversion rate:   {p_control:.2%}")
print(f"Treatment conversion rate: {p_treatment:.2%}")
print(f"Absolute lift:             {absolute_lift:.2%} ({absolute_lift*100:.2f} pp)")
print(f"Relative lift:             {relative_lift:.2%}")


Control conversion rate:   12.00%
Treatment conversion rate: 14.50%
Absolute lift:             2.50% (2.50 pp)
Relative lift:             20.83%


In [4]:
# Two-proportion z-test (two-sided)
x_control = int(control["registrations"])
x_treatment = int(treatment["registrations"])
n_control = int(control["users"])
n_treatment = int(treatment["users"])

pooled_rate = (x_control + x_treatment) / (n_control + n_treatment)
se_null = math.sqrt(
    pooled_rate * (1 - pooled_rate) * (1/n_control + 1/n_treatment)
)
z_score = absolute_lift / se_null
p_value = 2 * (1 - norm.cdf(abs(z_score)))

# 95% confidence interval for difference in proportions
se_ci = math.sqrt(
    p_control*(1-p_control)/n_control +
    p_treatment*(1-p_treatment)/n_treatment
)
ci_low = absolute_lift - 1.96 * se_ci
ci_high = absolute_lift + 1.96 * se_ci

print(f"z-score: {z_score:.3f}")
print(f"p-value: {p_value:.8f}")
print(f"95% CI for absolute lift: [{ci_low*100:.2f}, {ci_high*100:.2f}] percentage points")


z-score: 5.214
p-value: 0.00000018
95% CI for absolute lift: [1.56, 3.44] percentage points


In [5]:
# Guardrail analysis
error_control = control["page_error_rate"]
error_treatment = treatment["page_error_rate"]
error_diff = error_treatment - error_control

err_pooled = (control["page_errors"] + treatment["page_errors"]) / (n_control + n_treatment)
err_se = math.sqrt(err_pooled * (1-err_pooled) * (1/n_control + 1/n_treatment))
err_z = error_diff / err_se
err_p_value = 2 * (1 - norm.cdf(abs(err_z)))

print(f"Control page-error rate:   {error_control:.2%}")
print(f"Treatment page-error rate: {error_treatment:.2%}")
print(f"Difference:                {error_diff*100:.2f} pp")
print(f"Guardrail p-value:         {err_p_value:.4f}")
print("Guardrail threshold: no worse than +0.50 pp")


Control page-error rate:   1.00%
Treatment page-error rate: 1.10%
Difference:                0.10 pp
Guardrail p-value:         0.4879
Guardrail threshold: no worse than +0.50 pp


### Results

| Metric | Control | Treatment | Difference |
|---|---:|---:|---:|
| Users | 10,000 | 10,000 | — |
| Registration conversion | 12.00% | 14.50% | **+2.50 pp** |
| Relative lift | — | — | **+20.83%** |
| Page error rate | 1.00% | 1.10% | **+0.10 pp** |

**Statistical result**
- 95% confidence interval for the registration lift: **[1.56, 3.44] percentage points**
- Two-sided p-value: **0.00000018**
- The result is statistically significant at α = 0.05.
- The estimated +2.50 pp effect exceeds the +1.50 pp practical threshold (MDE).
- The full 95% CI is positive, and its lower bound is slightly above the practical threshold.
- The guardrail increase is only +0.10 pp, below the allowed +0.50 pp threshold.

Therefore, the result is both **statistically significant** and **practically meaningful** under the experiment's decision rule.


## 4. Causal Reasoning

Randomization helps estimate a causal effect because, in expectation, it balances both observed and unobserved user characteristics across Treatment and Control. This makes the landing-page version the main systematic difference between groups.

Because assignment occurs before the outcome and is random, the difference in registration conversion can be interpreted as an estimate of the causal effect of the redesigned landing page, assuming:
- users remain in their assigned group,
- there is no meaningful interference between users,
- tracking is implemented consistently,
- the experiment is run long enough to avoid short-lived novelty effects.

### Example confounder in an observational study
If the company showed the new page only to mobile users and the old page mostly to desktop users, **device type** could be a confounder. Mobile and desktop users may naturally have different conversion rates, so a simple comparison could mistakenly attribute a device effect to the new landing page.


## 5. Decision

### Decision: LAUNCH

The Treatment should be launched because:

1. **Effect size:** Registration increased from 12.0% to 14.5%, an absolute lift of **+2.5 percentage points**.
2. **Relative lift:** This is a **+20.8% relative improvement**.
3. **Statistical evidence:** The p-value is far below 0.05, and the 95% confidence interval excludes zero.
4. **Practical threshold:** The estimated effect exceeds the predefined **+1.5 pp MDE**, and the confidence interval lower bound is also above that threshold.
5. **Guardrail:** The page-error increase is only **+0.10 pp**, below the allowed **+0.50 pp** threshold.

### Limitation / assumption
The dataset is simulated for educational purposes. In a real production experiment, the team should also check sample-ratio mismatch, repeated-user assignment, seasonality, bot traffic, instrumentation quality, and longer-term downstream behavior.


## 6. Workflow

![Experiment workflow](workflow_diagram.png)

**Problem → Hypothesis → Treatment/Control → OEC & Guardrail → Analysis → Decision**


## 7. Final Conclusion

The redesigned landing page produced a meaningful increase in registrations without violating the technical guardrail. Under the predefined decision criteria, the evidence supports launching the Treatment.

### Tools Used
- Python
- pandas
- NumPy
- SciPy
- Jupyter Notebook
- GitHub

### SDAIA Academy
https://github.com/SDAIAAcademy
